In [8]:
import requests
import pandas as pd
import re

In [ ]:
def isTableText(text: str) -> bool:
    """
    Check if the given text contains table-like data.

    Args:
        text (str): The text to check.

    Returns:
        bool: True if the text appears to be a table, otherwise False.
    """
    if not isinstance(text, str):
        return False

    patterns = [
        r'<table\b',                # HTML table
        r'\n\s*\|',                 # markdown table rows
        r'^\s*\|.*\|\s*$',          # single markdown table line
        r'\n\s*-{2,}\s*\n',         # separator lines
        r'\t',                      # tab-separated table-like text
    ]

    return any(re.search(pattern, text, re.IGNORECASE | re.MULTILINE) for pattern in patterns) #Check if any of the patterns match the text, indicating it may be a table


def extractTitleText(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame containing only the title and text columns,
    excluding rows where text appears to be a table.

    Args:
        df (pd.DataFrame): The DataFrame containing the data.

    Returns:
        pd.DataFrame: A new DataFrame with only title and text columns.
    """
    result = df.loc[:, ['title', 'text']].copy() #Cuts down the dataframe to only the title and text columns
    result['text'] = result['text'].astype(str) #Removes any non-string values from the text column
    mask = result['text'].apply(lambda x: not isTableText(x)) #Creates a mask to filter out rows where the text appears to be a table
    return result.loc[mask, ['title', 'text']].reset_index(drop=True)

In [9]:
params = {
        'limit': '100',
        'logbook': 'SPEAR3'
}

r = requests.get('https://mccelog.slac.stanford.edu/elog/wbin/elog_display_json.php', params=params)


In [ ]:
data = r.json()
df = pd.DataFrame(data)

,title,text
0,"Tuning between fills. LTB Corrs, K3 pwr, Pkg....",
1,Shift Start: Thursday Swing Shift,"<table border=""1"" cellpadding=""0"" cellspacing=..."
2,Shift Summary: Thursday Day,Summary:\n8 hours of user beam delivered with ...
3,"Fill rates have been bouncy, however, BTS toro...",
4,9am meeting notes,
5,Shift Start: Thursday Day,"<table border=""1"" cellpadding=""1"" cellspacing=..."
6,Shift Summary: Thursday Owl,SUMMARY:\nUser beam delivered all 8 hours this...
7,Shift Summary: Thursday Owl,SUMMARY:\nUser beam delivered all 8 hours this...
8,Shift Summary: Thursday Owl,SUMMARY:\nUser beam delivered all 8 hours this...
9,Reference of LTB/BTS BPMs and injection effici...,


In [21]:
cleanDF = extractTitleText(df)
cleanDF

,title,text
0,"Tuning between fills. LTB Corrs, K3 pwr, Pkg....",
1,Shift Summary: Thursday Day,Summary:\n8 hours of user beam delivered with ...
2,"Fill rates have been bouncy, however, BTS toro...",
3,9am meeting notes,
4,Shift Summary: Thursday Owl,SUMMARY:\nUser beam delivered all 8 hours this...
5,Shift Summary: Thursday Owl,SUMMARY:\nUser beam delivered all 8 hours this...
6,Shift Summary: Thursday Owl,SUMMARY:\nUser beam delivered all 8 hours this...
7,Reference of LTB/BTS BPMs and injection effici...,
8,DO addressed this--problem was with the 4-3 LN...,
9,BL04 closed. Priority 2 Dewar fault MPS alarm....,
